In [ ]:
from AIce.models import linear
from AIce.functions import torchRMSE,trainloader,testloader,redim,getRMSE
from torch import optim

import numpy as np
import torch
import matplotlib.pyplot as plt

from time import strftime
# [['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm'],
#             ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos'],
#             ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath'],
#             ['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE'],
#             ['u_ERA5','v_ERA5','h_piomas','sic_CDR'],
#             ['u_ERA5','v_ERA5','h_piomas','sic_CDR', 'windnorm'],
#             ['u_ERA5','v_ERA5','h_piomas'],
#             ['u_ERA5','v_ERA5']]
device='cuda'

In [ ]:
# inputlist=['windnorm']
# lr =1e-3
# n_epoch=20



# _,_,dataloader,_,_,_,labels=trainloader('../data/DRIFT_DATA_TRAIN.csv',256,inputlist=inputlist,target='buoynorm',)
# mlp256=linear(len(labels[1])).to(device)


# print('Total Epoch: ', n_epoch)
# print('Learning Rate: ', lr)
# print(labels)

# rlosses={}
# losses=[]
# rloss=0
# rcount=0

# optimizer = optim.Adam(mlp256.parameters(),lr=lr) # optimizers

# for epoch in range(n_epoch):
#     print('Starting Epoch ',epoch)
#     curpercent=0
#     rl=[]
#     for i,data in enumerate(dataloader):

#         # get the inputs and the target
#         truth,inputs= data[0].to(device),data[1].to(device)

#         # zero the gradients,
#         optimizer.zero_grad()

#         # Forward,
#         out=mlp256(inputs)


#         loss = torchRMSE(target=truth, outputs=out)
#         # Backward
#         loss.backward()
#         losses.append(loss.item())
#         # Optimize

#         optimizer.step()

#         rloss+= loss.item()
#         rcount +=1
#         #print where we at plus the loss
#         percent= round(i/len(dataloader)*100)
#         if percent != curpercent:
#         #    print(percent, 'loss: ', rloss/rcount)
#             curpercent=percent
#             rl.append(rloss/rcount)
#             rloss=0
#             rcount=0
#     rlosses[epoch]=rl
#     print('Epoch avg loss: ', np.mean(rl))

# saving_name=f'linear_{len(labels[1])}inputs_{n_epoch}E_{lr}lr_{strftime("%d-%Hh_%Mm_%Ss")}'

# torch.save(mlp256.state_dict(), f'.//weights//{saving_name}.pt')

# with open(f'.//weights//{saving_name}.txt','w') as f:
#     f.write(f'Model: linear (on gpu)\n')
#     f.write(f'Weights: {saving_name}.pt\n')
#     f.write(f'Associated inputs: {inputlist}')

# fig,ax =plt.subplots()
# epochavg=[]

# for i in (rlosses):
#     x=np.arange(i*len(rlosses[0]),len(rlosses[0])+i*len(rlosses[0]))
#     ax.plot(x,rlosses[i])#,label=f'Epoch {i+1}, lr={lr[i]:.1e}')
#     avg=np.mean(rlosses[i])
#     epochavg.append(avg)
# xx=[i*len(rlosses[0]) for i in rlosses]
# ax.plot(xx,epochavg,'.-k')
# plt.savefig(f'.//outputs//loss_for_{saving_name}.png')
# plt.close('all')

Target is buoynorm normalized by log1p
Windnorm normalized by z-score
Total Epoch:  20
Learning Rate:  0.001
[Index(['buoynorm'], dtype='object'), Index(['windnorm'], dtype='object')]
Starting Epoch  0
Epoch avg loss:  1.1461702411138748
Starting Epoch  1
Epoch avg loss:  0.7977351072659857
Starting Epoch  2
Epoch avg loss:  0.7895665712572718
Starting Epoch  3
Epoch avg loss:  0.7896624259798082
Starting Epoch  4
Epoch avg loss:  0.7896993575449828
Starting Epoch  5
Epoch avg loss:  0.7897132226693762
Starting Epoch  6
Epoch avg loss:  0.7896706800048169
Starting Epoch  7
Epoch avg loss:  0.7896715539726583
Starting Epoch  8
Epoch avg loss:  0.7895060176312271
Starting Epoch  9
Epoch avg loss:  0.7896311413128296
Starting Epoch  10
Epoch avg loss:  0.7897337223049048
Starting Epoch  11
Epoch avg loss:  0.7896368845445768
Starting Epoch  12
Epoch avg loss:  0.7895650000919353
Starting Epoch  13
Epoch avg loss:  0.7896667576916925
Starting Epoch  14
Epoch avg loss:  0.7897125230418456
S

In [64]:
list(mlp256.parameters())

[Parameter containing:
 tensor([[0.4310]], device='cuda:0', requires_grad=True),
 Parameter containing:
 tensor([1.8898], device='cuda:0', requires_grad=True)]

In [ ]:
inputlist=['u_ERA5','v_ERA5','h_piomas','sic_CDR','x_EASE','y_EASE','bath','sin','cos', 'windnorm']
inputlist=['windnorm']
testdata,_,dataloader,means,stds,maxes,_=testloader('../data/DRIFT_DATA_TEST.csv',inputlist
                                    ,'buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible
linearmodel= linear(len(inputlist))
linearmodel.load_state_dict(torch.load('./weights/linear_10inputs_20E_0.001lr_09-17h_11m_05s.pt', weights_only= True))
linearmodel.eval()

out=linearmodel(torch.tensor(testdata.drop(['buoynorm'],axis=1).values, dtype= torch.float32))
testdata=redim(testdata,
               means=means,
               stds=stds,
               maxes=maxes)

testdata['linear']=np.expm1(out.detach().numpy())



rmse=getRMSE(testdata['buoynorm'].values, testdata['linear'].values)

print('RMSE: ', rmse)
from AIce.plotting import ResidualOnTheMap,PredAgainstTarget,histogrammeur

PredAgainstTarget(testdata['buoynorm'],testdata['linear'],testdata['windnorm'])
histogrammeur(testdata,testdata['buoynorm'],['linear'])

fig, ax =plt.subplots(dpi=200)
n_bins=200
ax.hist(testdata['buoynorm'], n_bins, color= "#C52C4684", label='Observed Distribution')
ax.hist(testdata['linear'], n_bins, histtype='step', label='linear')



ax.legend()
#plt.xlim([-np.pi,np.pi])
ax.set_xlim([-0,40])
ax.set_ylabel(f'Frequence bin={n_bins}')
ax.set_xlabel('Buoynorm [cm/s]')
ax.legend()
plt.show()


ResidualOnTheMap(testdata,pred='linear',target='buoynorm')


TypeError: 'DataLoader' object is not callable

In [51]:

testdata,_,dataloader,means,stds,maxes,labels=testloader('../data/DRIFT_DATA_TEST.csv',inputlist
                                    ,'buoynorm',trainingset_loaded=False, training_file='../data/DRIFT_DATA_TRAIN.csv')# Get the means,stds,maxes using the longest list possible
linearmodel= linear(len(inputlist))
linearmodel.load_state_dict(torch.load('./weights/linear_10inputs_20E_0.001lr_09-17h_11m_05s.pt', weights_only= True))
linearmodel.eval()

out=linearmodel(torch.tensor(testdata.drop(['buoynorm'],axis=1).values, dtype= torch.float32))

testsample=next(iter(dataloader))

(print(testdata.columns,labels))
print(testdata.loc[testdata.index==0])
print(testsample[0][0], testsample[1][0])
print(list(linearmodel.parameters()))

Bath size is the full test set
Target is buoynorm normalized by log1p
Sin and Cos added
Bathymetry (bath) normalized by maximum
x/y (x_EASE, y_EASE) normalized by maximum
Windnorm normalized by z-score
Wind components (u/v_ERA5) normalized by z-score
Index(['buoynorm', 'x_EASE', 'y_EASE', 'u_ERA5', 'v_ERA5', 'sic_CDR',
       'h_piomas', 'sin', 'cos', 'bath', 'windnorm'],
      dtype='object') [Index(['buoynorm'], dtype='object'), Index(['x_EASE', 'y_EASE', 'u_ERA5', 'v_ERA5', 'sic_CDR', 'h_piomas', 'sin',
       'cos', 'bath', 'windnorm'],
      dtype='object')]
   buoynorm    x_EASE    y_EASE    u_ERA5    v_ERA5   sic_CDR  h_piomas  \
0  1.811923  0.555987  0.485198  0.587591  0.190512  0.996916  2.090739   

        sin       cos     bath  windnorm  
0 -0.900969 -0.433884 -0.82566 -0.999713  
tensor([3.0645]) tensor([ 0.5488,  0.5703, -2.9757,  0.7084,  1.0000,  2.6010,  0.2393, -0.9709,
        -0.7117,  2.9998])
[Parameter containing:
tensor([[ 1.2613,  0.1793,  0.0551,  0.0508,  

tensor([2.7962]) tensor([ 0.5870,  0.3987,  0.8479, -0.1461,  1.0000,  1.7344,  0.1547, -0.9880,
        -0.0574, -0.6673])
[Parameter containing:
tensor([[ 1.2613,  0.1793,  0.0551,  0.0508,  0.3719, -0.2558, -0.1344, -0.1639,
         -0.5702,  0.4288]], requires_grad=True), Parameter containing:
tensor([0.9394], requires_grad=True)


In [46]:
list(linearmodel.parameters())

[Parameter containing:
 tensor([[ 1.2613,  0.1793,  0.0551,  0.0508,  0.3719, -0.2558, -0.1344, -0.1639,
          -0.5702,  0.4288]], requires_grad=True),
 Parameter containing:
 tensor([0.9394], requires_grad=True)]

In [61]:
nonameparam=[ 1.2613,  0.1793,  0.0551,  0.0508,  0.3719, -0.2558, -0.1344, -0.1639,
         -0.5702,  0.4288]
param={labels[1][i]: nonameparam[i] for i in range(labels[1].shape[0])}
bias=0.9394
sum=0
buoy_n=44
for i in param.keys():
    sum+=(testdata[i][buoy_n]*param[i])

print(sum, sum+bias, testdata['buoynorm'][buoy_n])


1.1936767230863274 2.1330767230863277 2.047247532745504
